# 🤖 Step 3 — Train Fraud Detection Model

We train a LightGBM classifier to predict fraud.
The model learns patterns from labeled historical transactions.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# Load Silver features as Pandas for sklearn
df = spark.table('silver_fraud_features').toPandas()

features = ['Amount', 'TimeSinceLastTxnMins', 'NumTxnLast24h',
            'AvgTxnAmount30d', 'location_jump', 'high_velocity',
            'amount_spike', 'fast_repeat']
target = 'IsFraud'

X = df[features]
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f'Training on {len(X_train)} samples, testing on {len(X_test)}')

In [ ]:
# Train model and log with MLflow
with mlflow.start_run(run_name='fraud_detection_v1'):
    model = GradientBoostingClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    mlflow.log_metric('accuracy', accuracy)
    mlflow.sklearn.log_model(model, 'fraud_model')
    
    print(f'\n✅ Model Accuracy: {accuracy:.1%}')
    print('\nDetailed Report:')
    print(classification_report(y_test, y_pred, target_names=['Legitimate','Fraud']))